# CS336 GPU Kernels & Triton — T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/CS336_tmp/blob/main/cs336_gpu_kernels_triton_t4.ipynb)

Stanford CS336 Spring 2026의 Lecture 5–6 + Assignment 2를 T4용으로 재구성했다.

반복 흐름: **PyTorch baseline → benchmark/profile → Triton → correctness → benchmark/profile → PTX → 병목/개선**

범위:
- Lecture 6 직접 구현: GeLU, Softmax, Row Sum, MatMul+ReLU
- Assignment 2: fused RMSNorm
- Lecture 5: FlashAttention 아이디어를 forward-only Triton kernel로 구현

공식 자료: https://github.com/stanford-cs336/lectures · https://github.com/stanford-cs336/assignment2-systems

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])

import math, torch, torch.nn.functional as F, triton, triton.language as tl
from torch.profiler import ProfilerActivity
assert torch.cuda.is_available()
device="cuda"
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(), torch.cuda.get_device_capability())

In [ ]:
def bench(fn, warmup=10, trials=30):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); ts=[]
    for _ in range(trials):
        a,b=torch.cuda.Event(True),torch.cuda.Event(True)
        a.record(); fn(); b.record(); torch.cuda.synchronize()
        ts.append(a.elapsed_time(b))
    return sum(ts)/len(ts)

def prof(fn, rows=10):
    for _ in range(3): fn()
    torch.cuda.synchronize()
    with torch.profiler.profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA]) as p:
        fn(); torch.cuda.synchronize()
    print(p.key_averages().table(sort_by="cuda_time_total",row_limit=rows,max_name_column_width=90))

def ptx(handle, memory_only=False, n=80):
    lines=handle.asm["ptx"].splitlines()
    if memory_only:
        keys=("ld.global","st.global","%ctaid","%tid",".reg")
        lines=[x for x in lines if any(k in x for k in keys)]
    print("\n".join(lines[:n]))

def compare(d):
    for name,fn in d.items():
        try: print(f"{name:28s} {bench(fn):8.4f} ms")
        except Exception as e: print(name,"ERROR",e)

## 1. GeLU — elementwise / fusion / 첫 PTX

In [ ]:
def gelu_naive(x): return .5*x*(1+torch.erf(x/math.sqrt(2)))
try: gelu_compiled=torch.compile(gelu_naive)
except Exception: gelu_compiled=None

@triton.jit
def gelu_k(x,y,n:tl.constexpr,B:tl.constexpr):
    o=tl.program_id(0)*B+tl.arange(0,B); m=o<n
    z=tl.load(x+o,mask=m)
    tl.store(y+o,.5*z*(1+tl.erf(z*.7071067811865476)),mask=m)

def gelu_triton(x,ret=False):
    y=torch.empty_like(x); B=256
    h=gelu_k[(triton.cdiv(x.numel(),B),)](x,y,x.numel(),B=B)
    return (y,h) if ret else y

x=torch.randn(4_000_000,device=device)
y,h_gelu=gelu_triton(x,True)
torch.testing.assert_close(y,gelu_naive(x),rtol=1e-4,atol=1e-5)

print("naive profile"); prof(lambda:gelu_naive(x))
print("builtin profile"); prof(lambda:F.gelu(x))
compare({"naive":lambda:gelu_naive(x),"builtin":lambda:F.gelu(x),"triton":lambda:gelu_triton(x)})
if gelu_compiled:
    try: gelu_compiled(x); compare({"torch.compile":lambda:gelu_compiled(x)})
    except Exception as e: print("compile skipped",e)

In [ ]:
ptx(h_gelu,n=100)

**확인:** naive의 여러 primitive kernel/HBM 왕복과 fused kernel을 비교한다. 첫 PTX에서는 global load/store, block/thread, register를 직접 본다.

## 2. Softmax — row reduction + fusion

In [ ]:
def softmax_naive(x):
    m=x.max(-1,keepdim=True).values; e=torch.exp(x-m); return e/e.sum(-1,keepdim=True)

@triton.jit
def softmax_k(x,y,C:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); o=tl.arange(0,B); m=o<C
    z=tl.load(x+r*C+o,mask=m,other=-float("inf"))
    z=z-tl.max(z,axis=0); e=tl.exp(z); out=e/tl.sum(e,axis=0)
    tl.store(y+r*C+o,out,mask=m)

def softmax_triton(x,ret=False):
    R,C=x.shape; B=triton.next_power_of_2(C); y=torch.empty_like(x)
    h=softmax_k[(R,)](x,y,C,B=B,num_warps=8 if B>=2048 else 4)
    return (y,h) if ret else y

xs=torch.randn(4096,1024,device=device)
ys,h_soft=softmax_triton(xs,True)
torch.testing.assert_close(ys,torch.softmax(xs,-1),rtol=2e-4,atol=2e-5)
print("naive"); prof(lambda:softmax_naive(xs))
print("triton"); prof(lambda:softmax_triton(xs))
compare({"naive":lambda:softmax_naive(xs),"builtin":lambda:torch.softmax(xs,-1),"triton":lambda:softmax_triton(xs)})
ptx(h_soft,True)

**확인:** `max→exp→sum→divide`가 여러 kernel인지, row 하나를 한 Triton program에 넣어 fuse하면 무엇이 줄어드는지 본다.

## 3. Row Sum — baby tiling

In [ ]:
@triton.jit
def rowsum_k(x,y,C:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); acc=tl.zeros((B,),tl.float32)
    for s in range(0,C,B):
        o=s+tl.arange(0,B); acc+=tl.load(x+r*C+o,mask=o<C,other=0.)
    tl.store(y+r,tl.sum(acc,axis=0))

def rowsum_triton(x,B=1024,ret=False):
    R,C=x.shape; y=torch.empty(R,device=x.device,dtype=torch.float32)
    h=rowsum_k[(R,)](x,y,C,B=B,num_warps=8)
    return (y,h) if ret else y

xr=torch.randn(1024,16384,device=device)
yr,h_row=rowsum_triton(xr,ret=True)
torch.testing.assert_close(yr,xr.sum(-1),rtol=2e-4,atol=2e-3)
for B in [256,512,1024,2048]: print(B,bench(lambda B=B:rowsum_triton(xr,B)))
compare({"torch.sum":lambda:xr.sum(-1),"triton":lambda:rowsum_triton(xr)})
ptx(h_row,True)

**확인:** row가 block보다 클 때 tile을 순회한다. `BLOCK_SIZE` sweep으로 tile 크기와 latency 변화를 본다.

## 4. MatMul + ReLU — tiling / compute / epilogue fusion

In [ ]:
@triton.jit
def mm_k(a,b,c,M:tl.constexpr,N:tl.constexpr,K:tl.constexpr,
         sam:tl.constexpr,sak:tl.constexpr,sbk:tl.constexpr,sbn:tl.constexpr,
         scm:tl.constexpr,scn:tl.constexpr,
         BM:tl.constexpr,BN:tl.constexpr,BK:tl.constexpr):
    pm,pn=tl.program_id(0),tl.program_id(1)
    om=pm*BM+tl.arange(0,BM); on=pn*BN+tl.arange(0,BN); ok=tl.arange(0,BK)
    ap=a+om[:,None]*sam+ok[None,:]*sak
    bp=b+ok[:,None]*sbk+on[None,:]*sbn
    acc=tl.zeros((BM,BN),tl.float32)
    for ks in range(0,K,BK):
        av=tl.load(ap,mask=(om[:,None]<M)&(ks+ok[None,:]<K),other=0.)
        bv=tl.load(bp,mask=(ks+ok[:,None]<K)&(on[None,:]<N),other=0.)
        acc+=tl.dot(av,bv); ap+=BK*sak; bp+=BK*sbk
    acc=tl.maximum(acc,0.)
    cp=c+om[:,None]*scm+on[None,:]*scn
    tl.store(cp,acc,mask=(om[:,None]<M)&(on[None,:]<N))

def mm_triton(a,b,BM=32,BN=32,BK=32,ret=False):
    M,K=a.shape; N=b.shape[1]; c=torch.empty((M,N),device=a.device,dtype=torch.float32)
    h=mm_k[(triton.cdiv(M,BM),triton.cdiv(N,BN))](
        a,b,c,M,N,K,*a.stride(),*b.stride(),*c.stride(),BM=BM,BN=BN,BK=BK,num_warps=4)
    return (c,h) if ret else c

a=torch.randn(1024,1024,device=device,dtype=torch.float16)
b=torch.randn(1024,1024,device=device,dtype=torch.float16)
c,h_mm=mm_triton(a,b,ret=True)
torch.testing.assert_close(c,torch.relu(a@b).float(),rtol=2e-2,atol=2e-1)
compare({"torch matmul+relu":lambda:torch.relu(a@b),"triton tiled+relu":lambda:mm_triton(a,b)})
for cfg in [(16,16,32),(32,32,32),(64,32,32)]:
    print(cfg,bench(lambda cfg=cfg:mm_triton(a,b,*cfg)))
ptx(h_mm,True)

**확인:** tiling은 A/B tile을 재사용해 arithmetic intensity를 높인다. ReLU를 output write 전에 붙여 별도 kernel/HBM 왕복을 없앤다.

## 5. RMSNorm — Assignment 2 style fused memory-bound kernel

In [ ]:
def rms_naive(x,w,eps=1e-6):
    v=x.float().pow(2).mean(-1,keepdim=True)
    return x*torch.rsqrt(v+eps).to(x.dtype)*w

@triton.jit
def rms_k(x,w,y,C:tl.constexpr,eps:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); o=tl.arange(0,B); m=o<C
    z=tl.load(x+r*C+o,mask=m,other=0.).to(tl.float32)
    ww=tl.load(w+o,mask=m,other=0.).to(tl.float32)
    inv=tl.rsqrt(tl.sum(z*z,axis=0)/C+eps)
    tl.store(y+r*C+o,z*inv*ww,mask=m)

def rms_triton(x,w,eps=1e-6,ret=False):
    R,C=x.shape; B=triton.next_power_of_2(C); y=torch.empty_like(x)
    h=rms_k[(R,)](x,w,y,C,eps,B=B,num_warps=8 if B>=2048 else 4)
    return (y,h) if ret else y

xx=torch.randn(4096,1024,device=device,dtype=torch.float16)
ww=torch.randn(1024,device=device,dtype=torch.float16)
yy,h_rms=rms_triton(xx,ww,ret=True)
torch.testing.assert_close(yy,rms_naive(xx,ww),rtol=3e-3,atol=3e-3)
print("naive"); prof(lambda:rms_naive(xx,ww))
print("triton"); prof(lambda:rms_triton(xx,ww))
compare({"naive RMSNorm":lambda:rms_naive(xx,ww),"triton fused":lambda:rms_triton(xx,ww)})
ptx(h_rms,True)

**확인:** RMSNorm은 FLOPs보다 memory movement가 중요할 수 있다. primitive kernel 수와 fused kernel 수를 profiler로 비교한다.

## 6. FlashAttention-style forward — tiled matmul + online softmax

In [ ]:
def attn_naive(q,k,v):
    s=(q@k.transpose(-2,-1))/math.sqrt(q.shape[-1])
    return torch.softmax(s,-1)@v

@triton.jit
def flash_k(q,k,v,o,sqb:tl.constexpr,sqh:tl.constexpr,sqn:tl.constexpr,sqd:tl.constexpr,
            skb:tl.constexpr,skh:tl.constexpr,skn:tl.constexpr,skd:tl.constexpr,
            svb:tl.constexpr,svh:tl.constexpr,svn:tl.constexpr,svd:tl.constexpr,
            sob:tl.constexpr,soh:tl.constexpr,son:tl.constexpr,sod:tl.constexpr,
            H:tl.constexpr,N:tl.constexpr,D:tl.constexpr,S:tl.constexpr,
            BM:tl.constexpr,BN:tl.constexpr):
    pm,bh=tl.program_id(0),tl.program_id(1); b=bh//H; h=bh%H
    om=pm*BM+tl.arange(0,BM); on=tl.arange(0,BN); od=tl.arange(0,D)
    qp=q+b*sqb+h*sqh+om[:,None]*sqn+od[None,:]*sqd
    qb=tl.load(qp,mask=(om[:,None]<N)&(od[None,:]<D),other=0.)
    mi=tl.full((BM,),-float("inf"),tl.float32); li=tl.zeros((BM,),tl.float32)
    acc=tl.zeros((BM,D),tl.float32)
    for st in range(0,N,BN):
        cn=st+on
        kp=k+b*skb+h*skh+cn[:,None]*skn+od[None,:]*skd
        vp=v+b*svb+h*svh+cn[:,None]*svn+od[None,:]*svd
        mask=(cn[:,None]<N)&(od[None,:]<D)
        kb=tl.load(kp,mask=mask,other=0.); vb=tl.load(vp,mask=mask,other=0.)
        qk=tl.dot(qb,tl.trans(kb))*S
        qk=tl.where((om[:,None]<N)&(cn[None,:]<N),qk,-float("inf"))
        mij=tl.maximum(mi,tl.max(qk,axis=1)); alpha=tl.exp(mi-mij)
        p=tl.exp(qk-mij[:,None]); lij=tl.sum(p,axis=1)
        acc=acc*alpha[:,None]+tl.dot(p.to(tl.float16),vb)
        li=li*alpha+lij; mi=mij
    out=acc/li[:,None]
    op=o+b*sob+h*soh+om[:,None]*son+od[None,:]*sod
    tl.store(op,out,mask=(om[:,None]<N)&(od[None,:]<D))

def flash_triton(q,k,v,BM=32,BN=32,ret=False):
    B,H,N,D=q.shape; o=torch.empty_like(q)
    hnd=flash_k[(triton.cdiv(N,BM),B*H)](
        q,k,v,o,*q.stride(),*k.stride(),*v.stride(),*o.stride(),
        H,N,D,1/math.sqrt(D),BM=BM,BN=BN,num_warps=4)
    return (o,hnd) if ret else o

q=torch.randn(1,4,256,64,device=device,dtype=torch.float16); k=torch.randn_like(q); v=torch.randn_like(q)
oo,h_flash=flash_triton(q,k,v,ret=True)
torch.testing.assert_close(oo,attn_naive(q,k,v),rtol=2e-2,atol=2e-2)
print("naive"); prof(lambda:attn_naive(q,k,v))
print("flash-style"); prof(lambda:flash_triton(q,k,v))
compare({"naive attention":lambda:attn_naive(q,k,v),"triton flash-style":lambda:flash_triton(q,k,v)})
ptx(h_flash,True)

**확인:** naive는 \(QK^T\) score와 softmax probability를 HBM에 materialize한다. FlashAttention-style은 Q tile을 잡고 K/V tile을 순회하며 online softmax를 갱신해 전체 \(N\times N\) 중간 tensor를 쓰지 않는다.

## 반복 체크표

| 연산 | 핵심 |
|---|---|
| GeLU | elementwise + fusion + PTX |
| Softmax | reduction + fusion |
| Row Sum | baby tiling |
| MatMul+ReLU | tiled reuse + epilogue fusion |
| RMSNorm | memory-bound fused kernel |
| Attention | tiled matmul + online softmax + HBM traffic 제거 |

각 항목에서 마지막 질문은 동일하다: **실제로 어떤 GPU kernel이 실행됐고, 병목은 무엇이었으며, custom kernel이 무엇을 줄였는가?**